# Retail Sales Pipeline: End-to-End Data Cleaning & Analytics Engine

**Author:** Mirza Ishtiyaq Baig *(Data Analyst / Analytics Engineer)*  
**Architecture:** Production-Grade Modular Data Pipeline (Ingestion -> Cleansing -> Integration -> Analytics -> Reporting)

---

## Pipeline Architecture Overview
This notebook implements a modular, production-ready data cleaning and analytics workflow for multi-source retail datasets (`customers.csv`, `orders.csv`, `transactions.csv`). It addresses raw data quality challenges including primary key deduplication, categorical string normalization, missing value imputation, and fulfillment SLA anomaly detection.

## 1. System Setup & Dependencies

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set global plotting parameters for production visualizations
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

# Define raw data file paths
DATA_DIR = '../data/raw/'
REPORTS_DIR = '../reports/figures/'
os.makedirs(REPORTS_DIR, exist_ok=True)

print('Dependencies loaded successfully.')

## 2. Data Ingestion & Schema Auditing

In [ ]:
# Ingest raw CSV extracts
customers_raw = pd.read_csv(os.path.join(DATA_DIR, 'customers.csv'))
orders_raw = pd.read_csv(os.path.join(DATA_DIR, 'orders.csv'))
transactions_raw = pd.read_csv(os.path.join(DATA_DIR, 'transactions.csv'))

print('Ingestion Audit:')
print(f'Raw Customers Shape   : {customers_raw.shape}')
print(f'Raw Orders Shape      : {orders_raw.shape}')
print(f'Raw Transactions Shape: {transactions_raw.shape}')

## 3. Modular Data Cleansing Engine

In [ ]:
def deduplicate_entities(df: pd.DataFrame, id_col: str) -> pd.DataFrame:
    """Deduplicate entity records based on primary key constraints."""
    initial_count = len(df)
    cleaned_df = df.drop_duplicates(subset=[id_col], keep='first').copy()
    dropped_count = initial_count - len(cleaned_df)
    print(f'[{id_col}] Deduplication: {initial_count} -> {len(cleaned_df)} rows ({dropped_count} duplicates removed)')
    return cleaned_df

# Execute primary key deduplication
customers_clean = deduplicate_entities(customers_raw, 'customer_id')
orders_clean = deduplicate_entities(orders_raw, 'order_id')
transactions_clean = deduplicate_entities(transactions_raw, 'transaction_id')

In [ ]:
# Impute missing contact attributes without dropping revenue data
customers_clean['email'] = customers_clean['email'].fillna('Not Provided')
customers_clean['phone_primary'] = customers_clean['phone_primary'].fillna('Not Provided')
customers_clean['phone_secondary'] = customers_clean['phone_secondary'].fillna('Not Provided')
customers_clean['city'] = customers_clean['city'].fillna('Unknown')

orders_clean['email'] = orders_clean['email'].fillna('Not Provided')

print('Missing value imputation completed.')

In [ ]:
# Categorical string standardization map for geographic attributes
country_map = {
    'United States': 'USA', 'usa': 'USA', 'US': 'USA', 'U.S.A.': 'USA', 'UNITED STATES': 'USA',
    'United Kingdom': 'UK', 'United kingdom': 'UK', 'united kingdom': 'UK', 'Great Britain': 'UK',
    'United Arab Emirates': 'UAE'
}

customers_clean['country'] = customers_clean['country'].replace(country_map)
print('Categorical country standardization completed. Unique countries:')
print(customers_clean['country'].unique())

## 4. Relational Data Integration & Schema Alignment

In [ ]:
# Resolve column name collision prior to relational merge
transactions_clean = transactions_clean.rename(columns={'purchase_amount': 'transaction_amount'})

# Relational Left Joins
customers_orders = pd.merge(customers_clean, orders_clean, on='customer_id', how='left')
full_data = pd.merge(customers_orders, transactions_clean, on='customer_id', how='left')

print(f'Master Analytical Schema Shape: {full_data.shape}')
print('Columns in Master Dataset:', full_data.columns.tolist())

## 5. Temporal Lead-Time Arithmetic & Anomaly Detection

In [ ]:
# Convert temporal strings to datetime objects
full_data['order_date'] = pd.to_datetime(full_data['order_date'])
full_data['ship_date'] = pd.to_datetime(full_data['ship_date'])

# Calculate fulfillment lead-time duration in days
full_data['ship_days'] = (full_data['ship_date'] - full_data['order_date']).dt.days

# Flag negative lead-time anomalies
valid_shipments = full_data[full_data['ship_days'] >= 0]
anomalous_shipments = full_data[full_data['ship_days'] < 0]

print(f'Total Master Records   : {len(full_data)}')
print(f'Valid Fulfillment Logs  : {len(valid_shipments)}')
print(f'Flagged Negative Days  : {len(anomalous_shipments)}')
print(f'Clean Avg Shipping Lead: {valid_shipments["ship_days"].mean():.1f} days')

## 6. Business Key Metrics & Exploratory Analytics

In [ ]:
# Business KPI Aggregations
total_revenue = full_data['total_amount'].sum()
avg_order_val = full_data['total_amount'].mean()
avg_transaction_val = full_data['transaction_amount'].mean()
total_units_sold = full_data['quantity'].sum()

print('=== EXECUTIVE FINANCIAL SUMMARY ===')
print(f'Total Gross Revenue       : ${total_revenue:,.2f}')
print(f'Average Order Value (AOV) : ${avg_order_val:,.2f}')
print(f'Average Transaction Value : ${avg_transaction_val:,.2f}')
print(f'Total Units Sold          : {total_units_sold:,}')

In [ ]:
# Top 5 Customers by Revenue
top_customers = (full_data.groupby('customer_name')['total_amount']
                          .sum()
                          .sort_values(ascending=False)
                          .head(5)
                          .reset_index())
top_customers.columns = ['Customer', 'Total Spent']

# Revenue Breakdown by Country
country_revenue = (full_data.groupby('country')['total_amount']
                            .sum()
                            .sort_values(ascending=False)
                            .reset_index())
country_revenue.columns = ['Country', 'Total Revenue']

# Monthly Revenue Velocity
full_data['financial_month'] = full_data['order_date'].dt.to_period('M')
monthly_revenue = (full_data.groupby('financial_month')['total_amount']
                             .sum()
                             .reset_index())
monthly_revenue.columns = ['Month', 'Monthly Revenue']

print('Top 5 Spenders:')
print(top_customers)
print('\nRevenue by Country:')
print(country_revenue)

## 7. Executive Data Visualization Dashboard

In [ ]:
# Construct 2x2 Production Dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Retail Sales & Supply Chain Performance Dashboard', fontsize=16, fontweight='bold')

# Chart 1: Top 5 Spenders
axes[0, 0].barh(top_customers['Customer'], top_customers['Total Spent'], color='#2b5c8f')
axes[0, 0].set_title('Top 5 Customers by Lifetime Spend', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Total Spent ($)')
axes[0, 0].invert_yaxis()
axes[0, 0].grid(axis='x', linestyle='--', alpha=0.5)

# Chart 2: Revenue by Country
axes[0, 1].bar(country_revenue['Country'], country_revenue['Total Revenue'], color='#d95f02')
axes[0, 1].set_title('Geographic Revenue Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Total Revenue ($)')
axes[0, 1].tick_params(axis='x', rotation=30)
axes[0, 1].grid(axis='y', linestyle='--', alpha=0.5)

# Chart 3: Monthly Revenue Velocity
months_str = monthly_revenue['Month'].astype(str)
axes[1, 0].plot(months_str, monthly_revenue['Monthly Revenue'], marker='o', color='#7570b3', linewidth=2)
axes[1, 0].set_title('Monthly Revenue Trajectory', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Monthly Revenue ($)')
axes[1, 0].grid(True, linestyle='--', alpha=0.5)

# Chart 4: Shipping Days Distribution
axes[1, 1].hist(valid_shipments['ship_days'], bins=8, color='#1b9e77', edgecolor='black', alpha=0.8)
axes[1, 1].set_title('Fulfillment Lead-Time Distribution (Valid Records)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Shipping Duration (Days)')
axes[1, 1].set_ylabel('Order Count')
axes[1, 1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
output_path = os.path.join(REPORTS_DIR, 'sales_pipeline_summary.png')
plt.savefig(output_path, dpi=300)
print(f'Executive dashboard visualization saved successfully to: {output_path}')
plt.show()